# Experimental Results Analysis and Comparison

This notebook loads and analyzes results from different federated learning strategies.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Analysis libraries imported successfully")

## 1. Load Experimental Results

In [ ]:
# Load results from experiments
results_dir = Path('../results/metrics')

results = {}
result_files = [
    ('baseline', 'baseline_results.json'),
    ('FedAvg', 'fedavg_results.json'),
    ('FedProx', 'fedprox_results.json'),
    ('FedDP', 'feddp_results.json')
]

for name, filename in result_files:
    filepath = results_dir / filename
    if filepath.exists():
        with open(filepath, 'r') as f:
            results[name] = json.load(f)
        print(f"Loaded {name}: {filename}")
    else:
        print(f"Not found: {filename}")

print(f"\nLoaded results for {len(results)} strategies")

## 2. Create Comparison Dataframe

In [ ]:
# Extract metrics for comparison
comparison_data = []

for strategy, result in results.items():
    metrics = result.get('final_metrics', {})
    comparison_data.append({
        'Strategy': strategy,
        'Accuracy': metrics.get('accuracy', 0),
        'Precision': metrics.get('precision', 0),
        'Recall': metrics.get('recall', 0),
        'F1-Score': metrics.get('f1', 0),
        'AUC-ROC': metrics.get('auc_roc', 0),
        'FPR': metrics.get('fpr', 0),
        'FNR': metrics.get('fnr', 0)
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nStrategy Comparison:")
print(comparison_df.to_string(index=False))

## 3. Metrics Comparison Visualization

In [ ]:
# Plot comparison
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#3498db', '#e74c3c', '#27ae60', '#f39c12']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    values = comparison_df[metric].values
    
    bars = ax.bar(comparison_df['Strategy'], values, color=colors, alpha=0.7, edgecolor='black')
    
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{value:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Training History Comparison

In [ ]:
# Plot training history for federated strategies
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

fed_strategies = [name for name in results.keys() if name != 'baseline']
colors_fed = ['#e74c3c', '#27ae60', '#f39c12']

for strategy, color in zip(fed_strategies, colors_fed):
    if 'training_history' in results[strategy]:
        history = results[strategy]['training_history']
        rounds = range(1, len(history.get('loss', [])) + 1)
        
        if 'loss' in history:
            ax1.plot(rounds, history['loss'], marker='o', label=strategy, color=color, linewidth=2)
        
        if 'accuracy' in history:
            ax2.plot(rounds, history['accuracy'], marker='s', label=strategy, color=color, linewidth=2)

ax1.set_xlabel('Round', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training Loss Over Rounds', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Round', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Accuracy Over Rounds', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Performance Analysis by Strategy

In [ ]:
print("\n" + "="*70)
print("DETAILED STRATEGY ANALYSIS")
print("="*70)

for strategy, result in results.items():
    print(f"\n{strategy.upper()}")
    print("-" * 70)
    
    metrics = result.get('final_metrics', {})
    cm = metrics.get('confusion_matrix', {})
    
    print(f"Configuration:")
    if 'num_clients' in result:
        print(f"  Clients: {result['num_clients']}")
    if 'num_rounds' in result:
        print(f"  Rounds: {result['num_rounds']}")
    if 'epochs' in result:
        print(f"  Epochs: {result['epochs']}")
    
    print(f"\nPerformance Metrics:")
    print(f"  Accuracy:    {metrics.get('accuracy', 0):.4f}")
    print(f"  Precision:   {metrics.get('precision', 0):.4f}")
    print(f"  Recall:      {metrics.get('recall', 0):.4f}")
    print(f"  F1-Score:    {metrics.get('f1', 0):.4f}")
    print(f"  Specificity: {metrics.get('specificity', 0):.4f}")
    
    print(f"\nConfusion Matrix:")
    print(f"  TP: {cm.get('TP', 0)}, FP: {cm.get('FP', 0)}")
    print(f"  FN: {cm.get('FN', 0)}, TN: {cm.get('TN', 0)}")

print("\n" + "="*70)

## 6. Radar Chart Comparison

In [ ]:
# Create radar chart
from math import pi

categories = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']
num_vars = len(categories)

angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

colors_radar = ['#3498db', '#e74c3c', '#27ae60', '#f39c12']

for idx, (strategy, color) in enumerate(zip(comparison_df['Strategy'], colors_radar)):
    values = [
        comparison_df.loc[comparison_df['Strategy'] == strategy, 'Accuracy'].values[0],
        comparison_df.loc[comparison_df['Strategy'] == strategy, 'Precision'].values[0],
        comparison_df.loc[comparison_df['Strategy'] == strategy, 'Recall'].values[0],
        comparison_df.loc[comparison_df['Strategy'] == strategy, 'F1-Score'].values[0],
    ]
    
    # Add specificity
    if 'FPR' in comparison_df.columns:
        specificity = 1 - comparison_df.loc[comparison_df['Strategy'] == strategy, 'FPR'].values[0]
        values.append(specificity)
    else:
        values.append(0.5)
    
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=strategy, color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=10)
ax.set_ylim(0, 1)
ax.set_title('Strategy Performance Comparison', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.show()

## 7. Statistical Summary

In [ ]:
print("\n" + "="*70)
print("STATISTICAL SUMMARY")
print("="*70)

# Best strategy for each metric
print("\nBest Performance by Metric:")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    best_idx = comparison_df[metric].idxmax()
    best_strategy = comparison_df.loc[best_idx, 'Strategy']
    best_value = comparison_df.loc[best_idx, metric]
    print(f"  {metric}: {best_strategy} ({best_value:.4f})")

# Comparison vs baseline
if 'baseline' in comparison_df['Strategy'].values:
    baseline_acc = comparison_df.loc[comparison_df['Strategy'] == 'baseline', 'Accuracy'].values[0]
    
    print(f"\nBaseline Accuracy: {baseline_acc:.4f}")
    print("\nImprovement over Baseline:")
    
    for strategy in comparison_df['Strategy']:
        if strategy != 'baseline':
            acc = comparison_df.loc[comparison_df['Strategy'] == strategy, 'Accuracy'].values[0]
            improvement = ((acc - baseline_acc) / baseline_acc) * 100
            print(f"  {strategy}: {improvement:+.2f}%")

print("\n" + "="*70)